## /data/.cryptomator

In [2]:
%%bash
FILE=/data/.cryptomator/auto-push.sh
mkdir -p $(dirname $FILE)

cat > $FILE << 'EOF'
#!/bin/bash
DIRS=(
  /data/.cryptomator
  /data/.manjaro
  /data/projects_ING/SMC
  /data/projects_ING/freqtrade_trade
)
for DIR in "${DIRS[@]}"; do
  if [[ -d "$DIR/.git" ]]; then
    echo "=== 正在处理: $DIR ==="
    cd "$DIR" && git add -A && git diff --cached --quiet || git commit -m "$(date '+%Y-%m-%d')" && git push
  else
    echo "=== 跳过: $DIR (不是 git 仓库或未挂载) ==="
  fi
done
EOF
chmod +x $FILE

ServiceFile=~/.config/systemd/user/git-backup.service
mkdir -p $(dirname $ServiceFile)
cat > $ServiceFile << EOF
[Unit]
Description=Git Auto Push
After=network.target

[Service]
ExecStart=$FILE
EOF

TimerFile=~/.config/systemd/user/git-backup.timer
cat > $TimerFile << 'EOF'
[Unit]
Description=Git Auto Push Daily

[Timer]
OnCalendar=*-*-* 00:00:00
Persistent=true

[Install]
WantedBy=timers.target
EOF

systemctl --user daemon-reload
systemctl --user enable --now $(basename $TimerFile)
systemctl --user status $(basename $TimerFile)
systemctl --user start $(basename $ServiceFile)
journalctl --user -u $(basename $ServiceFile) -n 50 --no-pager
systemctl --user list-timers --all


● git-backup.timer - Git Auto Push Daily
     Loaded: loaded (/home/kf/.config/systemd/user/git-backup.timer; enabled; preset: enabled)
     Active: active (waiting) since Thu 2026-04-23 22:05:41 UTC; 1min 4s ago
 Invocation: c818bc7471684fbfb56695ddae5cbfdd
    Trigger: Fri 2026-04-24 00:00:00 UTC; 1h 53min left
   Triggers: ● git-backup.service

Apr 23 22:05:41 kf-ms7d90 systemd[852]: Started Git Auto Push Daily.
Apr 23 22:05:41 kf-ms7d90 systemd[852]: Started Git Auto Push.
Apr 23 22:05:41 kf-ms7d90 auto-push.sh[23287]: === 正在处理: /data/.cryptomator ===
Apr 23 22:05:41 kf-ms7d90 auto-push.sh[23294]: [main bfa7389] 2026-04-23
Apr 23 22:05:41 kf-ms7d90 auto-push.sh[23294]:  14 files changed, 2 insertions(+), 11 deletions(-)
Apr 23 22:05:41 kf-ms7d90 auto-push.sh[23294]:  delete mode 100644 cryptomator/d/KC/QMKJGW25Y2YLSOAQ22NQYQ335VDN7C/ufdi8J0dqQQL2eHVT2JADdtBzmR32W8vXrVQCw0s.c9r
Apr 23 22:05:41 kf-ms7d90 auto-push.sh[23294]:  delete mode 100755 cryptomator/d/TV/KEETDNZK7AORGEWB6U76UF